# Replicate: CASIA-IVA-Lab FastSAM 
### pytorch checkpoint (.pt & .onnx)

In [ ]:
# New conda env

"""
# clean env wipe & activate
conda env remove --name FastSAM_onnx -y
conda create -n FastSAM_onnx python=3.10 -y
conda activate FastSAM_onnx

sudo apt update
sudo apt install libcudnn8 libcudnn8-dev
conda install -c conda-forge libstdcxx-ng=12 -y

# Instalation of FastSAM repo requirements
cd ~/FastSAM
pip install -r requirements.txt
python setup.py install
pip install timm==1.0.17 ultralytics==8.0.120 matplotlib==3.5.3


# typical np version mismatch
pip uninstall -y numpy
pip install numpy==1.26.4 

# Now install the CUDA PyTorch 2.5.0a0 (compatible with jp61 torch)
pip install --no-cache https://developer.download.nvidia.com/compute/redist/jp/v61/pytorch/torch-2.5.0a0+872d972e41.nv24.08.17622132-cp310-cp310-linux_aarch64.whl

# download torchvision 0.20.0a0 (compatible with jp61 torch)
cd ~/vision/
python3 setup.py install
cd ~

# Install ONNX runtime (general)
pip install onnx==1.13.1 onnxruntime==1.14.1 safetensors==0.4.1 sympy==1.13.1 onnxslim==0.1.59

# Install ONNX runtime for Jetson (GPU support)
wget https://nvidia.box.com/shared/static/48dtuob7meiw6ebgfsfqakc9vse62sg4.whl -O onnxruntime_gpu-1.16.3-cp310-cp310-linux_aarch64.whl
pip install onnxruntime_gpu-1.16.3-cp310-cp310-linux_aarch64.whl

# numpy conflict again !!
pip install "numpy>=1.19.2,<2.0"


pip install git+https://github.com/openai/CLIP.git

# Tensorrt
ls /usr/lib/python3.10/dist-packages/tensorrt/tensorrt.so
cp -r /usr/lib/python3.10/dist-packages/tensorrt* $CONDA_PREFIX/lib/python3.10/site-packages/
python -c "import tensorrt; print(tensorrt.__version__)"

"""

# Debug Support

# Check your conda env GLIBCXX versions
# strings /home/copter/miniconda3/envs/nanosam_arm64/lib/python3.10/site-packages/zmq/backend/cython/../../../../.././libstdc++.so.6 | grep GLIBCXX

# trt2torch is apparently an issue and is obsolete for trt --version=10.3.0 
# additionally `pip install pycuda`

In [ ]:
# ONNX FastSAM Inference Setup - Fix cuDNN version issue
import os
import sys
import torch
from ultralytics.yolo.engine.results import Results
from ultralytics.yolo.utils import ops
from random import randint
from typing import List, Tuple, Any
import time
import tensorrt     # import even if not used

# images and plotting
from PIL import Image
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import onnxruntime as ort

In [ ]:
# Verify versions

print(f"NumPy version: {np.__version__}")       # shold be 1.26.4 or else causes mem core dump issues
print(f"TensorRT version: {tensorrt.__version__}")
print("Available ONNX Runtime providers: ", ort.get_available_providers())

In [ ]:
# 🧹 Clear the PyTorch CUDA cache
torch.cuda.empty_cache()

import gc
gc.collect()

In [ ]:
# Add FastSAM utils if needed for compatibility
sys.path.append('/home/copter/FastSAM')

# Update library path to ensure cuDNN is found
os.environ['LD_LIBRARY_PATH'] = '/usr/lib/aarch64-linux-gnu:' + os.environ.get('LD_LIBRARY_PATH', '')

In [ ]:
# --- ONNX CONF ------------------- #

# Configure session options (for threading)
session_options = ort.SessionOptions()
session_options.intra_op_num_threads = 2
session_options.inter_op_num_threads = 1
# session_options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL


# Now try to set up all providers including CUDA
available_providers = ort.get_available_providers()
working_providers = []

# Test TensorRT first (best for Jetson)
if 'TensorrtExecutionProvider' in available_providers:
    print("✅ TensorRT provider available")
    trt_options = {
        'device_id': 0,
        'trt_max_workspace_size': 1 * 1024 * 1024 * 1024,  # 1GB
        'trt_fp16_enable': True,
        'trt_engine_cache_enable': True,
        'trt_timing_cache_enable': True,
    }
    working_providers.append(('TensorrtExecutionProvider', trt_options))

# Now try CUDA with cuDNN fix
if 'CUDAExecutionProvider' in available_providers:
    print("✅ CUDA provider available - testing with cuDNN fix...")
    cuda_options = {
        'device_id': 0,
        'arena_extend_strategy': 'kSameAsRequested',
        'gpu_mem_limit': 1 * 1024 * 1024 * 1024,  # 1GB
        'cudnn_conv_algo_search': 'HEURISTIC',
    }
    working_providers.append(('CUDAExecutionProvider', cuda_options))

# CPU fallback
cpu_options = {
    'intra_op_num_threads': 2,
    'inter_op_num_threads': 1,
    'enable_cpu_mem_arena': True,
}
working_providers.append(('CPUExecutionProvider', cpu_options))

print(f"Provider chain: {[p[0] if isinstance(p, tuple) else p for p in working_providers]}")
providers = working_providers

In [ ]:
def postprocess(preds, img, orig_imgs, retina_masks, conf, iou, agnostic_nms=False):
    """
    Post-processes the raw ONNX model predictions to generate segmentation masks and bounding boxes.

    Args:
        preds (list): Raw predictions from the ONNX model.
        img (np.ndarray): Pre-processed input image (resized and normalized).
        orig_imgs (np.ndarray or list): Original un-preprocessed image(s).
        retina_masks (bool): Whether to use retina masks (more detailed).
        conf (float): Confidence threshold for object detection.
        iou (float): IoU threshold for Non-Maximum Suppression (NMS).
        agnostic_nms (bool): Whether to perform class-agnostic NMS.

    Returns:
        list: A list of Results objects containing processed bounding boxes and masks.
    """
    p = ops.non_max_suppression(preds[0],
                                 conf,
                                 iou,
                                 agnostic_nms,
                                 max_det=100,
                                 nc=1) # nc=1 assuming a single class (e.g., 'object') for segmentation

    results = []
    # Adjusting for different ONNX output structures (e.g., if exported with specific opsets)
    proto = preds[1][-1] if len(preds[1]) == 3 else preds[1]
    
    for i, pred in enumerate(p):
        orig_img = orig_imgs[i] if isinstance(orig_imgs, list) else orig_imgs
        img_path = "inference_output" # Placeholder for image path

        if not len(pred):  # save empty boxes
            results.append(Results(orig_img=orig_img, path=img_path, names="segment", boxes=pred[:, :6]))
            continue

        if retina_masks:
            if not isinstance(orig_imgs, torch.Tensor):
                pred[:, :4] = ops.scale_boxes(img.shape[2:], pred[:, :4], orig_img.shape)
            masks = ops.process_mask_native(proto[i], pred[:, 6:], pred[:, :4], orig_img.shape[:2])  # HWC
        else:
            masks = ops.process_mask(proto[i], pred[:, 6:], pred[:, :4], img.shape[2:], upsample=True)  # HWC
            if not isinstance(orig_imgs, torch.Tensor):
                pred[:, :4] = ops.scale_boxes(img.shape[2:], pred[:, :4], orig_img.shape)
        
        results.append(
            Results(orig_img=orig_img, path=img_path, names="segment", boxes=pred[:, :6], masks=masks))
    return results

def pre_processing(img_origin, imgsz=1024):
    """
    Pre-processes an input image for the FastSAM ONNX model.

    Args:
        img_origin (np.ndarray): The original input image (BGR format).
        imgsz (int): The target image size (height and width) for the model.

    Returns:
        np.ndarray: The pre-processed image, ready for ONNX inference.
                    Shape: (1, 3, imgsz, imgsz) in RGB format, normalized to [0, 1].
    """
    h, w = img_origin.shape[:2]
    if h>w:
        scale   = min(imgsz / h, imgsz / w)
        inp     = np.zeros((imgsz, imgsz, 3), dtype = np.uint8)
        nw      = int(w * scale)
        nh      = int(h * scale)
        a = int((nh-nw)/2) 
        inp[: nh, a:a+nw, :] = cv2.resize(cv2.cvtColor(img_origin, cv2.COLOR_BGR2RGB), (nw, nh)) # <--- HERE
    else:
        scale   = min(imgsz / h, imgsz / w)
        inp     = np.zeros((imgsz, imgsz, 3), dtype = np.uint8)
        nw      = int(w * scale)
        nh      = int(h * scale)
        a = int((nw-nh)/2) 

        inp[a: a+nh, :nw, :] = cv2.resize(cv2.cvtColor(img_origin, cv2.COLOR_BGR2RGB), (nw, nh)) # <--- AND HERE
    rgb = np.array([inp], dtype = np.float32) / 255.0
    return np.transpose(rgb, (0, 3, 1, 2))

In [ ]:
# --- LOAD DOG IMAGE EXAMPLE (SKIP) ------------------------ #
image_path = "/home/copter/jetson_benchmark/images/dogs.jpg"
img_cv2 = cv2.imread(image_path)

if img_cv2 is None:
    raise FileNotFoundError(f"Image not found at {image_path}. Please check the path.")

print(f"Original Image Shape: {img_cv2.shape}")

# Pre-process the image for the ONNX model
inp = pre_processing(img_cv2)
print('Input shape for ONNX model:', inp.shape)

# Display the image
plt.figure(figsize=(10, 8))
# Convert the image from BGR to RGB before displaying with Matplotlib
plt.imshow(cv2.cvtColor(img_cv2, cv2.COLOR_BGR2RGB)) # <--- THIS IS THE FIX
plt.axis('on')
plt.title("Dogs Image (Corrected Colors)") # Changed title to reflect correction
plt.show()

In [ ]:
# --- LOAD VIDEO FRAME EXAMPLE ------------------------ #

# Part 1 -------------------------------- #

# video path (use for frame processing)
video_path = "/home/copter/Data/Aerial view of manhattan.mp4"

# Create a VideoCapture object
cap = cv2.VideoCapture(video_path)

# Check if video was opened successfully
if not cap.isOpened():
    print("Error: Could not open video file.")
    exit()

# Get video properties (frame width, height, and FPS)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# Define the codec and create a VideoWriter object to save the output
fourcc = cv2.VideoWriter_fourcc(*'mp4v') # You can change the codec
out = cv2.VideoWriter('/home/copter/Data/output_nyc_video.mp4', fourcc, fps, (frame_width, frame_height))


# Part 2 -------------------------------- #

# Read the first frame
ret, frame = cap.read()     # frame is a cv2 object in BGR format,

# Check if frame was read successfully
if ret:
    # Convert the frame to a PIL Image
    img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))       # img is a PIL object
    
    # Resize the image and run the encoder
    img = img.resize((1024, 1024), resample=Image.BILINEAR)
    img_arr = np.array(img).astype(np.float32) / 255.0                  # this is a nd.array
    input_image = img_arr.transpose(2, 0, 1)[None, :, :, :]
    
    # Display the first frame
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.title("First Frame")
    plt.axis('off')
    plt.show()

# Release the video objects
cap.release()

# Print image shape
print(f"Original Image Shape: {frame.shape}")  # (height, width, channels)

### Model Setup

In [ ]:
# FastSAM Processing Config vars
retina_masks = True
conf = 0.25
iou = 0.7
agnostic_nms = False

In [ ]:
# Pre-process the image for the ONNX model
print(f"Original Image Shape: {frame.shape}")

# Pre-process the image for the ONNX model
inp = pre_processing(frame)
print('Input shape for ONNX model:', inp.shape)

In [ ]:
# ONNX model configuration
model_path = "/home/copter/onnx_models/CASIA-IVA-Lab_FastSAM-s.onnx"

print("Current providers: ", providers)

try:
    if session_options != None:
        model = ort.InferenceSession(
            model_path, 
            providers=providers, 
            session_options=session_options
        )
    else:
        model = ort.InferenceSession(
            model_path, 
            providers=providers
        )
    print(f"ONNX model loaded successfully with {providers} (if available).")
except Exception as e:
    print(f"Could not load ONNX model with CUDAExecutionProvider, trying with CPUExecutionProvider: {e}")
    model = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])
    print("ONNX model loaded successfully with CPUExecutionProvider.")


### Single frame processing

In [ ]:
# ---- Inference with GPU MEM measurement [SKIP] ------------ #
import subprocess
import torch
import onnxruntime as ort
import numpy as np

def get_gpu_mem_nvidia_smi():
    out = subprocess.check_output(
        ["nvidia-smi",
         "--query-gpu=memory.used",
         "--format=csv,nounits,noheader"]
    )
    val = out.strip().split()[0]
    return int(val)  # may raise ValueError if val == b'[N/A]'

# 1) Load your session
model_path = "/home/copter/onnx_models/CASIA-IVA-Lab_FastSAM-s.onnx"
model = ort.InferenceSession(model_path,
                             providers=["CUDAExecutionProvider","CPUExecutionProvider"])

# 2) Prepare inputs
ort_inputs = {model.get_inputs()[0].name: inp}

# 3) Warm up & sync
torch.cuda.synchronize()

# 4) Try snapshot before inference
try:
    mem_before = get_gpu_mem_nvidia_smi()  # MiB
except Exception:
    mem_before = None

# 5) Setup CUDA events
start_evt = torch.cuda.Event(enable_timing=True)
end_evt   = torch.cuda.Event(enable_timing=True)

# 6) Run inference
start_evt.record()
preds = model.run(None, ort_inputs)
end_evt.record()

# 7) Wait for GPU
torch.cuda.synchronize()

# 8) Try snapshot after inference
try:
    mem_after = get_gpu_mem_nvidia_smi()   # MiB
except Exception:
    mem_after = None

# 9) Compute GPU timing
gpu_time_s = start_evt.elapsed_time(end_evt) / 1000.0

print(f"ONNX Runtime GPU inference time: {gpu_time_s:.3f} s")

# 10) Print GPU memory delta if both snapshots succeeded
if mem_before is not None and mem_after is not None:
    print(f"GPU memory increase: {mem_after - mem_before:.2f} MB")
else:
    print("GPU memory measurement unavailable; skipping delta.")

print(f"Raw ONNX prediction shapes: {[x.shape for x in preds]}")

# — Post‑processing as before —
predictions_for_postprocess = [
    torch.from_numpy(preds[0]),
    [
        [torch.from_numpy(preds[1]), torch.from_numpy(preds[2]), torch.from_numpy(preds[3])],
        torch.from_numpy(preds[4]),
        torch.from_numpy(preds[5])
    ]
]

result = postprocess(
    predictions_for_postprocess, 
    inp, 
    img, 
    retina_masks, 
    conf, 
    iou
)
masks  = result[0].masks.data
scores = result[0].boxes.data[:, 4].cpu().numpy()

print(f"Number of detected masks: {len(masks)}")
image_with_masks_final = np.copy(img)


In [ ]:
## ORIGINAL inference on one frame

# Prepare inputs for the ONNX model
ort_inputs = {model.get_inputs()[0].name: inp}

# Run inference
preds = model.run(None, ort_inputs)
print(f"Raw ONNX prediction shapes: {[x.shape for x in preds]}")

# Reconstructing `predictions_for_postprocess` as the `postprocess` function expects.
# This part is crucial and depends on your specific ONNX model export.
predictions_for_postprocess = [
    torch.from_numpy(preds[0]), # This is `preds[0]` in the `postprocess` function
    [
        [torch.from_numpy(preds[1]), torch.from_numpy(preds[2]), torch.from_numpy(preds[3])],
        torch.from_numpy(preds[4]),
        torch.from_numpy(preds[5])
    ] # This complex structure becomes `preds[1]` in the `postprocess` function
]

# Perform post-processing
result = postprocess(
    predictions_for_postprocess,
    inp, 
    frame, 
    retina_masks, 
    conf, 
    iou
)

# Extract masks and scores from the first result (assuming single image inference)
# Make sure your `result[0]` actually contains `masks.data` and `boxes.data`
# From the `postprocess` function: `boxes=pred[:, :6], masks=masks`
# So, `result[0].masks.data` should be available.
masks = result[0].masks.data # This is a torch.Tensor of shape (N, H, W) for N masks
# You also need scores to pick the best mask. Let's assume scores are in the 5th column of `pred` (index 4)
# from `ops.non_max_suppression`, which is then part of `result[0].boxes.data`.
scores = result[0].boxes.data[:, 4].cpu().numpy() # Extract scores and convert to numpy

print(f"Number of detected masks: {len(masks)}")

# Ensure image_with_masks_final is always defined
image_with_masks_final = np.copy(img) # Initialize with original image (BGR) in case no masks are found

In [ ]:
# --- Mask Visualization Logic (with conf thresh) ----------------
original_img = frame
conf_threshold = 0.45

if len(masks) > 0:
    # Get original image dimensions for resizing
    h_orig, w_orig = original_img.shape[:2]

    # Initialize a transparent canvas for all masks
    combined_mask_pil = Image.new("RGBA", (w_orig, h_orig), (0, 0, 0, 0))

    # Iterate through all masks and their scores
    for i in range(len(masks)):
        scores_flat = scores.flatten()
        mask_score = scores_flat[i]

        # 1) Check if the mask score is greater than the threshold
        if mask_score > conf_threshold:
            print(f"Mask idx: {i}   score: {mask_score:.3f}")

            # Convert the mask to numpy for PIL
            raw_mask_tensor = masks[i]
            if raw_mask_tensor.ndim == 3 and raw_mask_tensor.shape[0] == 1:
                raw_mask_tensor = raw_mask_tensor.squeeze(0) # Remove batch dimension if present

            raw_mask = raw_mask_tensor.cpu().numpy() # (H_mask,W_mask) in [0,1]

            # 2) Extract, threshold, and resize the mask
            binary = (raw_mask > 0.5).astype(np.uint8) * 255
            mask_img = Image.fromarray(binary)
            mask_rs = mask_img.resize((w_orig, h_orig), Image.BILINEAR) # Resize to original image size

            # 3) Build a 20%‑opaque red overlay
            alpha_mask = mask_rs.point(lambda p: int(p * 0.3)) # Scale to 0-51 (20% of 255)
            
            # Create a semi-transparent red layer for the current mask
            red_layer = Image.new("RGBA", (w_orig, h_orig), (0, 255, 0, 0))
            red_layer.putalpha(alpha_mask)

            # Overlay the current red layer onto the combined mask canvas
            combined_mask_pil = Image.alpha_composite(combined_mask_pil, red_layer)

    # Convert the original image (BGR to RGB) and then to PIL Image for overlaying
    img_rgb_pil = Image.fromarray(cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB))
    
    # Overlay the combined mask canvas onto the original image
    image_with_masks_pil = Image.alpha_composite(img_rgb_pil.convert("RGBA"), combined_mask_pil)
    
    # Convert back to OpenCV BGR format for saving with cv2.imwrite
    image_with_masks = cv2.cvtColor(np.array(image_with_masks_pil), cv2.COLOR_RGBA2BGR)
else:
    print("No masks detected to display.")
    image_with_masks = np.copy(original_img) # If no masks, just keep the original image

# Save the segmented image
output_path = "/home/copter/jetson_benchmark/output/FastSAM_onnx_output.png" # Update this path for your output
cv2.imwrite(output_path, image_with_masks)
print(f"Segmented image saved to {output_path}")

In [ ]:
# # --- Top Single Mask Visualization Logic [SKIP THIS]----------------
# # original_img = img_cv2
# original_img = frame


# if len(masks) > 0:
#     # 1) Pick the best mask based on score
#     scores_flat = scores.flatten()
#     best_idx    = int(np.argmax(scores_flat))
#     print(f"Best mask idx: {best_idx}   score: {scores_flat[best_idx]:.3f}")

#     # Convert the best mask to numpy for PIL
#     # Ensure mask_out is a 2D tensor (H, W) or convert if it's (1, H, W)
#     raw_mask_tensor = masks[best_idx]
#     if raw_mask_tensor.ndim == 3 and raw_mask_tensor.shape[0] == 1:
#         raw_mask_tensor = raw_mask_tensor.squeeze(0) # Remove batch dimension if present

#     raw_mask = raw_mask_tensor.cpu().numpy() # (H_mask,W_mask) in [0,1]

#     # Get original image dimensions for resizing
#     # h_orig, w_orig = img_cv2.shape[:2]
#     h_orig, w_orig = original_img.shape[:2]

#     # 2) Extract, threshold, and resize the mask
#     binary    = (raw_mask > 0.5).astype(np.uint8) * 255
#     mask_img  = Image.fromarray(binary)
#     mask_rs   = mask_img.resize((w_orig, h_orig), Image.BILINEAR) # Resize to original image size

#     # 3) Build a 20%‑opaque red overlay
#     alpha_mask = mask_rs.point(lambda p: int(p * 0.3)) # Scale to 0-51 (20% of 255)
#     red_layer  = Image.new("RGBA", (w_orig, h_orig), (0, 255, 0, 0))
#     red_layer.putalpha(alpha_mask)

#     # Convert the original image (BGR to RGB) and then to PIL Image for overlaying
#     img_rgb_pil = Image.fromarray(cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB))
    
#     # Overlay the red layer onto the original image
#     image_with_masks_pil = Image.alpha_composite(img_rgb_pil.convert("RGBA"), red_layer)
    
#     # Convert back to OpenCV BGR format for saving with cv2.imwrite
#     image_with_masks = cv2.cvtColor(np.array(image_with_masks_pil), cv2.COLOR_RGBA2BGR)
# else:
#     print("No masks detected to display.")
#     image_with_masks = np.copy(original_img) # If no masks, just keep the original image

# # Save the segmented image
# output_path = "/home/copter/jetson_benchmark/output/FastSAM_onnx_output.png" # Update this path for your output
# cv2.imwrite(output_path, image_with_masks)
# print(f"Segmented image saved to {output_path}")

In [ ]:
# Display the output open iamge 
display_img = cv2.imread(output_path)
if display_img is None:
    raise FileNotFoundError(f"Could not read the saved image from {output_path}")

# Display the image (optional, for Jupyter)
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
# Convert from BGR (cv2.imread default) to RGB for matplotlib display
plt.imshow(cv2.cvtColor(display_img, cv2.COLOR_BGR2RGB))
plt.axis('on')
plt.title("Segmented Mask using FastSAM.onnx")
plt.show()


### 🎬 Video Frame Annotation

In [ ]:
# --- Run inference on full video ------ #

import time
import os
from tqdm import tqdm

# Video processing setup
# video_path = "/home/copter/Data/Aerial view of manhattan.mp4"
# output_video_path = "/home/copter/Data/output_nyc_video_fastsam.mp4"

video_path = "/home/copter/Data/Cars are Traffic_Road Riding in Mexico City_Made with Clipchamp_1.mp4"
output_video_path = "/home/copter/Data/output_bikes_in_cdmx_demo_fastsam_03.mp4"

# Create VideoCapture object
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print("Error: Could not open video file.")
    exit()

# Get video properties
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Video properties: {frame_height}x{frame_width}, {fps} FPS, {total_frames} frames")

# Create VideoWriter for output
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

# Processing configuration
conf_threshold = 0.45
frame_count = 0
start_time = time.time()

print("Starting video processing...")

# Initialize tqdm progress bar
with tqdm(total=total_frames, desc="Processing video", unit="frame") as pbar:

    while True:
        ret, frame = cap.read()
        if not ret:
            print("End of video or failed to read frame")
            break

        if frame_count>250:
            print("End of video, manual cut...")
            break
        
        frame_count += 1
        
        
        # Progress indicator
        if frame_count % 30 == 0:  # Print every 30 frames
            elapsed = time.time() - start_time
            fps_processed = frame_count / elapsed
            eta = (total_frames - frame_count) / fps_processed if fps_processed > 0 else 0
            print('len masks: ', len(masks) )
            print(f"Processing frame {frame_count}/{total_frames} "
                f"({frame_count/total_frames*100:.1f}%) - "
                f"Speed: {fps_processed:.1f} FPS - ETA: {eta:.1f}s")
        
        try:
            # Convert frame to PIL and resize for FastSAM input
            # img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            # img_resized = img.resize((1024, 1024), resample=Image.BILINEAR)
            # img_arr = np.array(img_resized).astype(np.float32) / 255.0
            # inp = img_arr.transpose(2, 0, 1)[None, :, :, :]
            inp = pre_processing(frame, imgsz=1024)
            
            # Prepare ONNX inputs
            ort_inputs = {model.get_inputs()[0].name: inp}
            
            # Run FastSAM inference (main CPU/GPU...)
            preds = model.run(None, ort_inputs)
            
            # Reconstruct predictions for postprocessing
            predictions_for_postprocess = [
                torch.from_numpy(preds[0]),
                [
                    [torch.from_numpy(preds[1]), torch.from_numpy(preds[2]), torch.from_numpy(preds[3])],
                    torch.from_numpy(preds[4]),
                    torch.from_numpy(preds[5])
                ]
            ]
            
            # Perform post-processing
            result = postprocess(
                predictions_for_postprocess,
                inp,
                frame,
                retina_masks,
                conf,
                iou
            )
            
            # Extract masks and scores
            if len(result) > 0 and result[0].masks is not None:
                masks = result[0].masks.data
                scores = result[0].boxes.data[:, 4].cpu().numpy()
                
                # Apply mask visualization logic
                original_img = frame
                h_orig, w_orig = original_img.shape[:2]
                
                if len(masks) > 0:
                    # Initialize transparent canvas for all masks
                    combined_mask_pil = Image.new("RGBA", (w_orig, h_orig), (0, 0, 0, 0))

                    if frame_count % 30 == 0:  # Print every 30 frames
                        print('len masks: ', len(masks) )
                    
                    # Process each mask
                    for i in range(len(masks)):
                        scores_flat = scores.flatten()
                        mask_score = scores_flat[i]
                        
                        # Apply confidence threshold
                        if mask_score > conf_threshold:
                            # Convert mask to numpy and process
                            raw_mask_tensor = masks[i]
                            if raw_mask_tensor.ndim == 3 and raw_mask_tensor.shape[0] == 1:
                                raw_mask_tensor = raw_mask_tensor.squeeze(0)
                            raw_mask = raw_mask_tensor.cpu().numpy()
                            
                            # Create binary mask and resize
                            binary = (raw_mask > 0.5).astype(np.uint8) * 255
                            mask_img = Image.fromarray(binary)
                            mask_rs = mask_img.resize((w_orig, h_orig), Image.BILINEAR)
                            
                            # Create semi-transparent green overlay (30% opacity)
                            alpha_mask = mask_rs.point(lambda p: int(p * 0.3))
                            red_layer = Image.new("RGBA", (w_orig, h_orig), (0, 255, 0, 0))
                            red_layer.putalpha(alpha_mask)
                            
                            # Combine with existing masks
                            combined_mask_pil = Image.alpha_composite(combined_mask_pil, red_layer)
                    
                    # Overlay masks on original image
                    img_rgb_pil = Image.fromarray(cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB))
                    image_with_masks_pil = Image.alpha_composite(img_rgb_pil.convert("RGBA"), combined_mask_pil)
                    processed_frame = cv2.cvtColor(np.array(image_with_masks_pil), cv2.COLOR_RGBA2BGR)
                else:
                    # No masks detected, use original frame
                    processed_frame = np.copy(original_img)
            else:
                # No detection results, use original frame
                processed_frame = np.copy(frame)
                
        except Exception as e:
            print(f"Error processing frame {frame_count}: {e}")
            # Use original frame if processing failsr
            processed_frame = np.copy(frame)
        
        # Write processed frame to output video
        out.write(processed_frame)

# Clean up
cap.release()
out.release()

# Final statistics
end_time = time.time()
total_time = end_time - start_time
avg_fps = frame_count / total_time

print(f"\n--- Video Processing Complete ---")
print(f"Total frames processed: {frame_count}")
print(f"Total processing time: {total_time:.2f} seconds")
print(f"Average processing speed: {avg_fps:.2f} FPS")
print(f"Output video saved to: {output_video_path}")

# Verify output video
if os.path.exists(output_video_path):
    output_size = os.path.getsize(output_video_path) / (1024*1024)  # MB
    print(f"Output video size: {output_size:.1f} MB")
else:
    print("Warning: Output video file not found!")